usage: colab_kernel_launcher.py [-h] [--num_processes NUM_PROCESSES]
                                [--download_folder DOWNLOAD_FOLDER]
                                image_list
colab_kernel_launcher.py: error: unrecognized arguments: -f


SystemExit: 2

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# -----------------------------
# STEP 1: INSTALL REQUIRED LIBRARIES
# -----------------------------
# Installing the essential, stable libraries for our new approach.
!pip install moviepy==1.0.3 numpy==1.26.4
!pip install tensorflow==2.15.0 tensorflow_hub==0.16.1
!pip install librosa==0.10.1

import os
import numpy as np
import librosa
import tensorflow as tf
import tensorflow_hub as hub
from moviepy.editor import VideoFileClip
import warnings
import csv

# Suppress noisy warnings for a cleaner output
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

print("Libraries installed successfully.")

# -----------------------------
# STEP 2: LOAD THE YAMNET AUDIO CLASSIFIER
# -----------------------------
try:
    print("\nLoading Google's YAMNet audio classifier... (This is a one-time download)")
    yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

    # Load the class names that YAMNet can predict
    class_map_path = yamnet_model.class_map_path().numpy()
    class_names = []
    with tf.io.gfile.GFile(class_map_path) as csvfile:
        reader = csv.reader(csvfile)
        next(reader) # Skip header
        for row in reader:
            class_names.append(row[2])

    print("YAMNet model and class names loaded successfully.")
except Exception as e:
    print(f"\nFATAL ERROR: Could not load the YAMNet model. Please check your internet connection. Details: {e}")
    raise e

# -----------------------------
# STEP 3: CORE ANALYSIS FUNCTIONS (NEW AND IMPROVED)
# -----------------------------
def extract_audio(video_path, audio_path="temp_audio.wav"):
    """Extracts audio and prepares it for YAMNet (16kHz mono WAV)."""
    try:
        with VideoFileClip(video_path) as clip:
            clip.audio.write_audiofile(audio_path, fps=16000, nbytes=2, codec='pcm_s16le', logger=None)
        return audio_path
    except Exception as e:
        print(f"\nERROR: Could not extract audio from the video. Please check the file. Details: {e}")
        return None

def detect_segments_with_yamnet(audio_path):
    """
    Analyzes audio with YAMNet, classifies each segment as 'speech' or 'music',
    and merges the results into a clean timeline.
    """
    if not os.path.exists(audio_path): return []
    try:
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        if len(y) < 1: return []

        # Run the model on the entire audio file
        scores, embeddings, spectrogram = yamnet_model(y)
        predictions = np.argmax(scores, axis=1)

        # Get the class index for 'Speech' and 'Music'
        speech_index = class_names.index('Speech')
        music_index = class_names.index('Music')

        labels = []
        timestamps = []

        # YAMNet produces a prediction every 0.48 seconds
        time_step = 0.48

        for i, score_row in enumerate(scores):
            # Get the raw scores for our target classes
            speech_score = score_row[speech_index]
            music_score = score_row[music_index]

            # Simple decision: which score is higher?
            if speech_score > 0.1 or music_score > 0.1: # Ignore near-silence
                if speech_score > music_score:
                    current_label = 'speech'
                else:
                    current_label = 'music'

                labels.append(current_label)
                timestamps.append(i * time_step)

        if not labels: return []

        # Merge consecutive segments with the same label
        clean_segments = []
        start_time, current_label = timestamps[0], labels[0]

        for i in range(1, len(labels)):
            if labels[i] != current_label:
                # Add a small buffer to the end time to close gaps
                end_time = timestamps[i] + 0.5
                clean_segments.append({"start": start_time, "end": end_time, "label": current_label})
                start_time, current_label = timestamps[i], labels[i]

        clean_segments.append({"start": start_time, "end": len(y) / sr, "label": current_label})
        return clean_segments

    except Exception as e:
        print(f"\nERROR: An issue occurred during the YAMNet analysis. Details: {e}")
        return []

# -----------------------------
# STEP 4: MAIN FUNCTION TO RUN THE ANALYSIS
# -----------------------------
def analyze_video(video_file_path):
    """Runs the full, stable pipeline and prints the final results."""
    print("\n" + "="*50)
    if not os.path.exists(video_file_path):
        print(f"FATAL ERROR: The video file was not found at '{video_file_path}'")
        print("Please make sure the file is uploaded and the path is correct.")
        print("="*50)
        return

    print(f"Starting analysis for: {video_file_path}")

    audio_file = extract_audio(video_file_path)
    if not audio_file: return

    final_segments = detect_segments_with_yamnet(audio_file)

    if os.path.exists(audio_file):
        os.remove(audio_file)

    print("\n--- ✅ ANALYSIS COMPLETE ✅ ---")
    if not final_segments:
        print("No clear speech or music segments were detected.")
    else:
        print("Detected Segments:")
        for seg in final_segments:
            print(f"  {seg['start']:>7.2f} - {seg['end']:>7.2f} sec  →  {seg['label']}")
    print("="*50)

# ---------------------------------------------------------------------------
# >> PLEASE UPDATE THIS LINE WITH THE NAME OF YOUR UPLOADED VIDEO FILE <<
# ---------------------------------------------------------------------------
video_path = "/content/t2.mp4"
# ---------------------------------------------------------------------------

analyze_video(video_path)

ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0)
ERROR: No matching distribution found for tensorflow==2.15.0
Libraries installed successfully.

Loading Google's YAMNet audio classifier... (This is a one-time download)
YAMNet model and class names loaded successfully.

Starting analysis for: /content/t2.mp4

--- ✅ ANALYSIS COMPLETE ✅ ---
Detected Segments:
     0.00 -    9.14 sec  →  speech
     8.64 -   11.54 sec  →  music
    11.04 -   14.90 sec  →  speech
    14.40 -   15.53 sec  →  music


In [ ]:
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y
!python3.10 -m venv /usr/local/py310
!/usr/local/py310/bin/python -m pip install --upgrade pip
!/usr/local/py310/bin/python -m pip install TTS==0.22.0
!/usr/local/py310/bin/python -m pip install torch gradio moviepy torchaudio librosa openai-whisper dtw resemblyzer deep_translator numpy==1.22
!/usr/local/py310/bin/python -m pip install transformers==4.41.0
!/usr/local/py310/bin/python -m pip install sentence-transformers==2.2.2
!/usr/local/py310/bin/python -m pip install numpy==1.26.4
!/usr/local/py310/bin/python -m pip install tensorflow tensorflow_hub
!/usr/local/py310/bin/python -m pip install pandas

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 http://security.ubuntu.com/ubuntu jammy-security InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
# Install latest stable Gradio for general compatibility, if 4.26.0 fails, you can try pinning.
%%writefile vocasyncseg.py

# Import statements should come after successful installation
import os
import numpy as np
import librosa
import tensorflow as tf
import tensorflow_hub as hub
from moviepy.editor import VideoFileClip, AudioFileClip, concatenate_audioclips
import warnings
import csv
import gradio as gr
import shutil
import tempfile
import uuid
import torch
import soundfile as sf
import gc
from pathlib import Path
from TTS.api import TTS
from pydub import AudioSegment
import pandas as pd

# AI Models & Utils
import whisper
from deep_translator import GoogleTranslator
from scipy.spatial.distance import cosine
from resemblyzer import VoiceEncoder, preprocess_wav

# ==== XTTS GLOBAL FIX FOR PYTORCH SAFE UNPICKLING ====
# These imports are specific to XTTS configuration and should remain.
from TTS.tts.configs.xtts_config import XttsConfig, XttsArgs
from TTS.tts.models.xtts import XttsAudioConfig
from TTS.config.shared_configs import BaseDatasetConfig

torch.serialization.add_safe_globals([
    XttsConfig,
    XttsAudioConfig,
    BaseDatasetConfig,
    XttsArgs
])
# =================================================================

# --- CRITICAL: VERIFY GRADIO VERSION ---
# The printed version MUST be 4.26.0 if you uncommented --force-reinstall above.
# Otherwise, it will be the latest installed.
print("-" * 50)
print(f"VERIFICATION: Using Gradio version: {gr.__version__}")
# If you specifically need 4.26.0, adjust this message:
# print("This version MUST be 4.26.0 for the UI to be stable, if you pinned it.")
print("-" * 50)

# Suppress noisy warnings for a cleaner output
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')


# ==== GLOBAL CONFIG & MODEL LOADING ====
print("Setting up global configurations and loading models...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
TEMP_DIR = "temp_outputs"
os.makedirs(TEMP_DIR, exist_ok=True)

print("Loading TTS models (this may take a moment)...")
try:
    tts_models = {
        "YourTTS": TTS(model_name="tts_models/multilingual/multi-dataset/your_tts", progress_bar=False).to(device),
        "XTTS": TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device),
    }
    print("TTS models loaded successfully.")
except Exception as e:
    print(f"FATAL ERROR: Could not load TTS models. This might be due to an incompatible PyTorch/Python version or TTS installation issue. Details: {e}")
    # Consider raising an error or exiting here if TTS is critical
    raise e


# --- YAMNet Model Loading ---
try:
    print("\nLoading Google's YAMNet audio classifier... (This is a one-time download)")
    yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')
    class_map_path = yamnet_model.class_map_path().numpy()
    class_names = []
    with tf.io.gfile.GFile(class_map_path) as csvfile:
        reader = csv.reader(csvfile)
        next(reader)
        for row in reader:
            class_names.append(row[2])
    print("YAMNet model loaded successfully.")
except Exception as e:
    print(f"\nFATAL ERROR: Could not load the YAMNet model. Details: {e}")
    raise e

def clear_gpu_memory():
    """Attempts to clear GPU memory if CUDA is available."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("GPU memory cleared.")

def extract_audio(video_path: str, output_audio_path: str) -> str | None:
    """Extracts audio from a video file."""
    try:
        video = VideoFileClip(video_path)
        video.audio.write_audiofile(output_audio_path, logger=None)
        video.close()
        return output_audio_path
    except Exception as e:
        print(f"ERROR: Could not extract audio from video: {e}")
        return None

def replace_audio_in_video(video_path: str, new_audio_path: str, output_video_path: str):
    """Replaces the audio track of a video with a new audio file."""
    try:
        video_clip = VideoFileClip(video_path)
        audio_clip = AudioFileClip(new_audio_path)
        final_clip = video_clip.set_audio(audio_clip)
        final_clip.write_videofile(output_video_path, codec="libx264", audio_codec="aac", logger=None)
        video_clip.close()
        audio_clip.close()
        final_clip.close()
    except Exception as e:
        print(f"ERROR: Could not replace audio in video: {e}")

def pad_audio_to_duration(audio_path: str, target_duration_ms: int) -> AudioSegment:
    """Pads or truncates an audio segment to a target duration."""
    try:
        audio = AudioSegment.from_wav(audio_path)
        current_duration_ms = len(audio)
        if current_duration_ms < target_duration_ms:
            padding = AudioSegment.silent(duration=target_duration_ms - current_duration_ms)
            audio += padding
        elif current_duration_ms > target_duration_ms:
            audio = audio[:target_duration_ms]
        return audio
    except Exception as e:
        print(f"ERROR: Could not pad/truncate audio: {e}")
        return AudioSegment.silent(duration=target_duration_ms) # Return silent segment on error

def detect_segments_with_yamnet(audio_path: str, progress_bar) -> list[dict]:
    """Detects speech and music segments using YAMNet."""
    if not os.path.exists(audio_path):
        print(f"ERROR: Audio file not found at {audio_path}")
        return []
    if yamnet_model is None or not class_names:
        print("ERROR: YAMNet model or class names not loaded. Cannot detect segments.")
        return []

    try:
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        if y.size < 1: # Use .size for numpy arrays
            print("WARNING: Loaded audio has no samples.")
            return []

        progress_bar(0.3, desc="Analyzing audio with YAMNet model...")
        waveform = tf.constant(y, dtype=tf.float32)
        scores, embeddings, spectrogram = yamnet_model(waveform)

        # Ensure 'Speech' and 'Music' are in class_names
        if 'Speech' not in class_names or 'Music' not in class_names:
            print("ERROR: 'Speech' or 'Music' not found in YAMNet class names. Check YAMNet setup.")
            return []

        speech_index = class_names.index('Speech')
        music_index = class_names.index('Music')

        labels, timestamps = [], []
        time_step = 0.48 # YAMNet hop length in seconds

        for i, score_row in enumerate(scores.numpy()):
            speech_score = score_row[speech_index]
            music_score = score_row[music_index]

            if speech_score > 0.1 or music_score > 0.1:
                current_label = 'speech' if speech_score > music_score else 'music'
                labels.append(current_label)
                timestamps.append(i * time_step)

        if not labels:
            print("No significant speech or music segments detected by YAMNet.")
            return []

        progress_bar(0.8, desc="Merging segments...")
        clean_segments = []
        if timestamps:
            start_time, current_label = timestamps[0], labels[0]

            for i in range(1, len(labels)):
                # If the label changes or there's a significant gap, end the current segment
                if labels[i] != current_label: # A new segment starts
                    end_time = timestamps[i]
                    clean_segments.append({"start": start_time, "end": end_time, "label": current_label})
                    start_time, current_label = timestamps[i], labels[i]

            # Add the last segment
            clean_segments.append({"start": start_time, "end": len(y) / sr, "label": current_label})

        return clean_segments
    except Exception as e:
        print(f"\nERROR during YAMNet analysis: {e}")
        return []

def split_text_into_chunks(text: str, max_chars: int = 500) -> list[str]:
    """Splits text into chunks of maximum characters."""
    words = text.split()
    chunks: list[str] = []
    current_chunk: list[str] = []

    for word in words:
        # Check if adding the next word exceeds max_chars,
        # considering spaces between words
        if sum(len(w) for w in current_chunk) + len(word) + len(current_chunk) > max_chars:
            if current_chunk: # Only append if the chunk is not empty
                chunks.append(" ".join(current_chunk))
            current_chunk = []
        current_chunk.append(word)

    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

# ============================ MAIN PROCESSING FUNCTIONS ============================
def process_video_dubbing(
    video_path: str,
    source_language: str,
    target_language: str,
    whisper_model_size: str,
    tts_model_choice: str,
    progress=gr.Progress()
):
    if not video_path:
        raise gr.Error("❌ Video File is required.")
    if not os.path.exists(video_path):
        raise gr.Error(f"❌ Video File not found at: {video_path}")

    temp_dir = tempfile.mkdtemp()
    original_audio_path = os.path.join(temp_dir, "original_audio.wav")
    # Use uuid for unique filenames to prevent conflicts if multiple users
    final_dubbed_video_path = os.path.join(TEMP_DIR, f"final_dubbed_video_{uuid.uuid4()}.mp4")
    final_audio_path = os.path.join(TEMP_DIR, f"final_dubbed_audio_{uuid.uuid4()}.wav")

    # Initialize an empty DataFrame for early returns
    empty_df = pd.DataFrame(columns=["Start (s)", "End (s)", "Label"])

    try:
        # 1. Extract audio from video and get segments
        progress(0.05, desc="Extracting audio from video and segmenting...")
        extracted_audio_for_segmentation = extract_audio(video_path, original_audio_path)
        if not extracted_audio_for_segmentation:
            raise gr.Error("Error extracting audio from video.")

        segments = detect_segments_with_yamnet(extracted_audio_for_segmentation, progress)
        if not segments:
            return "No clear speech or music segments were detected for dubbing.", None, empty_df, None

        # Load original full audio
        original_full_audio = AudioSegment.from_wav(original_audio_path)
        video_clip = VideoFileClip(video_path)
        video_duration_ms = video_clip.duration * 1000

        speech_segments_to_process = [s for s in segments if s['label'] == 'speech']
        music_segments = [s for s in segments if s['label'] == 'music']

        if not speech_segments_to_process:
            return "No speech segments detected for dubbing.", None, empty_df, None

        # --- Transcription & Language Detection (only for speech segments) ---
        progress(0.2, desc="Loading Whisper model...")
        # Ensure whisper is imported and load_model is available
        import whisper
        whisper_model = whisper.load_model(whisper_model_size, device=device)

        all_transcribed_text = []
        dubbed_audio_parts = []

        # Ensure tts_model_choice is valid and TTS models are loaded
        if tts_model_choice not in tts_models or tts_models[tts_model_choice] is None:
            raise gr.Error(f"TTS model '{tts_model_choice}' not found or not loaded. Please check TTS setup.")
        tts_model = tts_models[tts_model_choice]

        for i, seg in enumerate(speech_segments_to_process):
            start_sec, end_sec = seg['start'], seg['end']
            duration_sec = end_sec - start_sec
            progress(0.2 + (0.6 * (i / len(speech_segments_to_process))), desc=f"Processing speech segment {i+1}/{len(speech_segments_to_process)}: Transcribing and Translating...")

            segment_audio = original_full_audio[int(start_sec * 1000) : int(end_sec * 1000)]
            segment_audio_path = os.path.join(temp_dir, f"speech_seg_{i}.wav")
            segment_audio.export(segment_audio_path, format="wav")

            transcription_result = whisper_model.transcribe(segment_audio_path, language=source_language, fp16=torch.cuda.is_available())
            transcribed_text = transcription_result["text"]
            all_transcribed_text.append(f"[{start_sec:.2f}-{end_sec:.2f}] {transcribed_text}")

            translated_text = GoogleTranslator(source="auto", target=target_language).translate(transcribed_text)

            out_wav_path = os.path.join(temp_dir, f"dubbed_seg_{i}.wav")
            try:
                # Ensure the TTS model is properly configured and its method is correct
                # This part is highly dependent on your TTS library (e.g., Coqui TTS)
                tts_model.tts_to_file(
                    text=translated_text,
                    file_path=out_wav_path,
                    speaker_wav=segment_audio_path, # This is for voice cloning/adaptation
                    language=target_language,
                    speed=1.0, temperature=0.7, length_penalty=1.0, repetition_penalty=2.0
                )
            except Exception as e:
                print(f"WARNING: Error during TTS for segment {i}: {e}. Generating silent segment.")
                silent_segment = AudioSegment.silent(duration=int(duration_sec * 1000))
                silent_segment.export(out_wav_path, format="wav")

            dubbed_segment_audio = pad_audio_to_duration(out_wav_path, int(duration_sec * 1000))
            dubbed_audio_parts.append({
                "start_ms": int(start_sec * 1000),
                "end_ms": int(end_sec * 1000),
                "audio_segment": dubbed_segment_audio
            })

        del whisper_model # Clean up model from memory
        clear_gpu_memory()

        progress(0.9, desc="Stitching all audio segments...")
        final_audio = AudioSegment.silent(duration=int(video_duration_ms))

        for part in dubbed_audio_parts:
            # Overlay with a gain adjustment if desired, e.g., -6dB
            final_audio = final_audio.overlay(part["audio_segment"], position=part["start_ms"])

        # Overlay music segments from original audio, possibly with reduced volume
        for seg in music_segments:
            start_ms, end_ms = int(seg['start'] * 1000), int(seg['end'] * 1000)
            music_segment = original_full_audio[start_ms:end_ms]
            # Reduce music volume when overlaying, e.g., by 10 dB
            final_audio = final_audio.overlay(music_segment - 10, position=start_ms)

        final_audio.export(final_audio_path, format="wav")
        print(f"Final stitched audio saved to: {final_audio_path}")

        progress(0.95, desc="Replacing video audio...")
        replace_audio_in_video(video_path, final_audio_path, final_dubbed_video_path)

        # Create a pandas DataFrame for the segments output
        # Ensure all strings are standard UTF-8. For "speech" and "music", this is inherently true.
        segments_data = [[s['start'], s['end'], s['label']] for s in segments]
        segments_df = pd.DataFrame(segments_data, columns=["Start (s)", "End (s)", "Label"])

        return "\n".join(all_transcribed_text), segments_df, final_audio_path

    except gr.Error as e:
        # Re-raise Gradio errors so they are displayed to the user
        raise e
    except Exception as e:
        # Catch any other unexpected errors during the process
        print(f"An unexpected error occurred during video dubbing: {e}")
        raise gr.Error(f"An unexpected error occurred: {e}. Please check the logs.")
    finally:
        # Clean up temporary directory
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
            print(f"Cleaned up temporary directory: {temp_dir}")
        # Ensure video_clip is closed if it was opened
        if 'video_clip' in locals() and video_clip.is_playing: # Check if it's open
            video_clip.close()
            print("Video clip closed.")
# ============================ GRADIO UI ============================
with gr.Blocks(theme=gr.themes.Soft(), title="VocaSync Suite") as ui:
    gr.Markdown("# 🗣 VocaSync Suite: Voice Cloning & Video Dubbing")
    gr.Markdown("A comprehensive tool for voice cloning, emotion transfer, and AI-powered video dubbing, powered by Coqui TTS.")
    gr.Markdown("This version integrates YAMNet for intelligent speech/music segmentation and combines dubbed speech with original background music.")

    with gr.Tabs():
        with gr.TabItem("▶️ Video Dubbing & Cloning"):
            gr.Markdown("### Upload a video to dub its speech while preserving background music.")

            with gr.Row():
                with gr.Column(scale=1):
                    video_input = gr.Video(label="🎬 Input Video", sources=["upload"])

                    with gr.Row():
                        source_language = gr.Dropdown(
                            label="Source Language of Speech",
                            choices=['en', 'hi', 'te', 'ta', 'ur', 'es', 'fr', 'de', 'it', 'pt', 'pl', 'ru', 'zh-cn', 'ja', 'ko', 'ar'],
                            value='en'
                        )
                        target_language = gr.Dropdown(
                            label="Target Language for Dubbing",
                            choices=['en', 'hi', 'te', 'ta', 'ur', 'es', 'fr', 'de', 'it', 'pt', 'pl', 'ru', 'zh-cn', 'ja', 'ko', 'ar'],
                            value='en'
                        )
                    with gr.Row():
                       whisper_model_dub = gr.Dropdown(
                            label="Whisper Model Size",
                            choices=["tiny", "base", "small", "medium", "large"],
                            value="base"
                        )
                       tts_model_dub = gr.Dropdown(
                            label="TTS Model for Cloning",
                            choices=["YourTTS", "XTTS"],
                            value="XTTS"
                        )
                    dub_btn = gr.Button("🚀 Start Video Dubbing", variant="primary")

                with gr.Column(scale=1):
                    transcribed_speech_output = gr.Textbox(
                        label="📝 Transcribed & Translated Speech Segments", interactive=False, lines=15
                    )
                    # Keep this line, the problem was with the Dataframe output, not the video output.
                    #dubbed_video_output = gr.Video(label="📺 Dubbed Video Output") # <-- ADDED THIS BACK
                    dubbed_audio_output = gr.Audio(label="🔊 Dubbed Audio Output (Speech + Music)", type="filepath")
                    segment_info_output = gr.Dataframe(
                        headers=["Start (s)", "End (s)", "Label"],
                        datatype=["number", "number", "str"],
                        label="Detected Audio Segments (Speech/Music)",
                        interactive=False,
                        # The key change: specify type="pandas" if you intend to pass a DataFrame directly
                        type="pandas" # Explicitly tell Gradio it will receive a pandas DataFrame
                    )

            dub_btn.click(
                fn=process_video_dubbing,
                inputs=[video_input, source_language, target_language, whisper_model_dub, tts_model_dub],
                # Ensure outputs match the return order of the function
                outputs=[transcribed_speech_output, segment_info_output, dubbed_audio_output] # Corrected outputs
            )

# ============================ GRADIO UI ============================
'''with gr.Blocks(theme=gr.themes.Soft(), title="VocaSync Suite") as ui:
    gr.Markdown("# 🗣 VocaSync Suite: Voice Cloning & Video Dubbing")
    gr.Markdown("A comprehensive tool for voice cloning, emotion transfer, and AI-powered video dubbing, powered by Coqui TTS.")
    gr.Markdown("This version integrates YAMNet for intelligent speech/music segmentation and combines dubbed speech with original background music.")

    with gr.Tabs():
        with gr.TabItem("▶️ Video Dubbing & Cloning"):
            gr.Markdown("### Upload a video to dub its speech while preserving background music.")

            with gr.Row():
                with gr.Column(scale=1):
                    video_input = gr.Video(label="🎬 Input Video", sources=["upload"])

                    with gr.Row():
                        source_language = gr.Dropdown(
                            label="Source Language of Speech",
                            choices=['en', 'hi', 'te', 'ta', 'ur', 'es', 'fr', 'de', 'it', 'pt', 'pl', 'ru', 'zh-cn', 'ja', 'ko', 'ar'], # Added more languages
                            value='en'
                        )
                        target_language = gr.Dropdown(
                            label="Target Language for Dubbing",
                            choices=['en', 'hi', 'te', 'ta', 'ur', 'es', 'fr', 'de', 'it', 'pt', 'pl', 'ru', 'zh-cn', 'ja', 'ko', 'ar'], # Added more languages
                            value='en'
                        )
                    with gr.Row():
                       whisper_model_dub = gr.Dropdown(
                            label="Whisper Model Size",
                            choices=["tiny", "base", "small", "medium"],
                            value="base"
                        )
                       tts_model_dub = gr.Dropdown(
                            label="TTS Model for Cloning",
                            choices=["YourTTS", "XTTS"],
                            value="XTTS"
                        )
                    dub_btn = gr.Button("🚀 Start Video Dubbing", variant="primary")

                with gr.Column(scale=1):
                    transcribed_speech_output = gr.Textbox(
                        label="📝 Transcribed & Translated Speech Segments", interactive=False, lines=15
                    )
                    #dubbed_video_output = gr.Video(label="📺 Dubbed Video Output")
                    dubbed_audio_output = gr.Audio(label="🔊 Dubbed Audio Output (Speech + Music)", type="filepath") # New Audio Output
                    segment_info_output = gr.Dataframe(
                        headers=["Start (s)", "End (s)", "Label"],
                        datatype=["number", "number", "str"],
                        label="Detected Audio Segments (Speech/Music)",
                        interactive=False
                    )

            dub_btn.click(
                fn=process_video_dubbing,
                inputs=[video_input, source_language, target_language, whisper_model_dub, tts_model_dub],
                outputs=[transcribed_speech_output, segment_info_output, dubbed_audio_output] # Updated outputs
            )'''

if __name__ == "__main__":
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR)
    os.makedirs(TEMP_DIR, exist_ok=True)
    ui.launch(share=True, debug=True)

Writing vocasyncseg.py


In [ ]:
!/usr/local/py310/bin/python vocasyncseg.py


2025-09-05 16:19:47.375886: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
--------------------------------------------------
VERIFICATION: Using Gradio version: 5.44.1
--------------------------------------------------
Setting up global configurations and loading models...
Using device: cuda
Loading TTS models (this may take a moment)...
 > tts_models/multilingual/multi-dataset/your_tts is already downloaded.
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | >